### Imports + Chargement

In [26]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [27]:
# Chargement du dataset fusionné
df = pd.read_csv("../data/application_merged.csv")

print("Shape initiale :", df.shape)
df.head()

Shape initiale : (307511, 184)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,CC_AMT_CREDIT_LIMIT_ACTUAL_mean,CC_AMT_CREDIT_LIMIT_ACTUAL_max,CC_AMT_DRAWINGS_CURRENT_mean,CC_AMT_DRAWINGS_CURRENT_sum,CC_CNT_DRAWINGS_CURRENT_mean,CC_CNT_DRAWINGS_CURRENT_sum,CC_SK_DPD_mean,CC_SK_DPD_max,CC_SK_DPD_DEF_mean,CC_SK_DPD_DEF_max
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,270000.0,270000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Séparation cible & variables

In [28]:
# Séparation de la cible
y = df["TARGET"]

# Suppression de la cible + identifiant
X = df.drop(columns=["TARGET", "SK_ID_CURR"]).copy()

### Corrections des anomalies connues

In [29]:
# Correction valeur aberrante dans DAYS_EMPLOYED (la valeur 365243 correspond à un code technique connu du dataset)
X["DAYS_EMPLOYED"] = X["DAYS_EMPLOYED"].replace(365243, np.nan)

### Création de features

##### Variables corrigées

In [30]:
# Âge du client en années
# DAYS_BIRTH est exprimé en jours négatifs avant la demande
X["AGE"] = -X["DAYS_BIRTH"] / 365

# Ancienneté professionnelle en années
# DAYS_EMPLOYED est aussi exprimé en jours négatifs
X["EMPLOYMENT_YEARS"] = -X["DAYS_EMPLOYED"] / 365

##### Ratios financiers

In [31]:
# Poids du crédit par rapport au revenu :
# plus ce ratio est élevé, plus le crédit peut être difficile à supporter
X["CREDIT_INCOME_RATIO"] = X["AMT_CREDIT"] / X["AMT_INCOME_TOTAL"]

In [32]:
# Charge de remboursement par rapport au revenu
# cette variable est plus informative que l'annuité seule
X["ANNUITY_INCOME_RATIO"] = X["AMT_ANNUITY"] / X["AMT_INCOME_TOTAL"]

In [33]:
# Taille approximative du foyer
# permet de relativiser le niveau de revenu du ménage
X["HOUSEHOLD_SIZE"] = X["CNT_FAM_MEMBERS"]

In [34]:
# Revenu par personne du foyer
# utile pour mieux représenter la capacité financière réelle
X["INCOME_PER_PERSON"] = X["AMT_INCOME_TOTAL"] / X["HOUSEHOLD_SIZE"]

##### Historique de crédit

In [35]:
# Part de dette dans le crédit bureau cumulé
# mesure le niveau d'endettement restant par rapport aux crédits historiques
X["DEBT_CREDIT_RATIO"] = X["BUREAU_AMT_CREDIT_SUM_DEBT_sum"] / X["BUREAU_AMT_CREDIT_SUM_sum"]

In [36]:
# Retard moyen observé sur les crédits POS/CASH
# indicateur comportemental de risque
X["POS_DPD_RATIO"] = X["POS_SK_DPD_mean"]

In [37]:
# Retard moyen observé sur les cartes de crédit
# autre signal de comportement de paiement
X["CC_DPD_RATIO"] = X["CC_SK_DPD_mean"]

In [38]:
# Taux d'utilisation moyen du crédit renouvelable
# plus il est élevé, plus le client semble dépendre de sa réserve de crédit
X["CC_UTILIZATION"] = X["CC_AMT_BALANCE_mean"] / X["CC_AMT_CREDIT_LIMIT_ACTUAL_mean"]

##### Demandes précédentes

In [39]:
# Rapport entre montant demandé et montant accordé
# peut refléter un historique de dossiers moins bien acceptés
X["APPROVED_CREDIT_RATIO"] = X["PREV_AMT_APPLICATION_mean"] / X["PREV_AMT_CREDIT_mean"]

##### Variables logarithmiques

In [40]:
# Certaines variables financières ont été identifiées comme très asymétriques dans l'EDA
# Création des versions log-transformées
log_cols = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_GOODS_PRICE",
    "BUREAU_AMT_CREDIT_SUM_sum"
]

for col in log_cols:
    X[col + "_LOG"] = np.log1p(X[col])

##### Indicateur global de qualité de ligne

In [41]:
# Nombre de valeurs manquantes par client
# cette variable peut parfois capturer un profil de dossier plus ou moins complet
X["MISSING_COUNT"] = X.isnull().sum(axis=1)

### Nettoyage post-feature engineering

In [42]:
# Traitement des valeurs infinies
X.replace([np.inf, -np.inf], np.nan, inplace=True)

In [43]:
# Réduction des redondances
X.drop(columns=["DAYS_BIRTH", "DAYS_EMPLOYED"], inplace=True) # AGE et EMPLOYMENT_YEARS remplacent DAYS_BIRTH et DAYS_EMPLOYED

### Séparation train/test

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Gestion des valeurs manquantes

In [45]:
# Identification des types de variables
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Variables numériques :", len(num_cols))
print("Variables catégorielles :", len(cat_cols))

Variables numériques : 180
Variables catégorielles : 16


In [46]:
# Imputation des variables numériques par la médiane
imputer_num = SimpleImputer(strategy="median")
X_train[num_cols] = imputer_num.fit_transform(X_train[num_cols])
X_test[num_cols] = imputer_num.transform(X_test[num_cols])

# Imputation des variables catégorielles par la modalité la plus fréquente
imputer_cat = SimpleImputer(strategy="most_frequent")
X_train[cat_cols] = imputer_cat.fit_transform(X_train[cat_cols])
X_test[cat_cols] = imputer_cat.transform(X_test[cat_cols])

### Encodage des variables catégorielles

In [47]:
# One-hot encoding sur les catégories restantes
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# Alignement des colonnes entre train et test
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

### Vérifications finales

In [48]:
print("NaN train :", X_train.isnull().sum().sum())
print("NaN test  :", X_test.isnull().sum().sum())

print("Shape train finale :", X_train.shape)
print("Shape test finale  :", X_test.shape)

NaN train : 0
NaN test  : 0
Shape train finale : (246008, 304)
Shape test finale  : (61503, 304)


### Normalisation des noms de colonnes

In [49]:
X_train.columns = X_train.columns.str.replace(r"[^A-Za-z0-9_]+", "_", regex=True)
X_test.columns = X_test.columns.str.replace(r"[^A-Za-z0-9_]+", "_", regex=True)

### Sauvegarde

In [50]:
joblib.dump(X_train, "../data/X_train.pkl")
joblib.dump(X_test, "../data/X_test.pkl")
joblib.dump(y_train, "../data/y_train.pkl")
joblib.dump(y_test, "../data/y_test.pkl")

['../data/y_test.pkl']

Les données ont été préparées pour la modélisation :

- gestion des valeurs manquantes
- création de variables métier
- encodage des variables catégorielles
- séparation train/test

Le dataset est désormais prêt pour l'entraînement des modèles.